# K5: evaluation + results tables

Generated by `scripts/make_notebooks.py`; see `docs/kaggle_runbook.md`. Run it headless:
**Save Version -> Save & Run All (Commit)**. No cell needs editing.

- **Accelerator:** None (CPU) for 50m; **switch to GPU for 500m**. **Internet:** ON.
- **Attach (Add Input):** the K2 output for SIZE, the K3 output, and the K4 output(s) whose test runs to evaluate
- **Expected time:** 50m < 1 h
- **Output (`/kaggle/working/out`):** `results/*-eval-*.json`, `*-peruser.npz`, `reports/results.md`

In [ ]:
# Parameters: the defaults work unedited (attach inputs, then Save Version ->
# Save & Run All). Generated by scripts/make_notebooks.py.
REPO_URL = "https://github.com/Arubad/hookseq.git"  # the HookSeq repo (private: add a GITHUB_TOKEN secret)
GIT_REF = "main"  # branch or tag to run (pin a tag for reproducible runs)
SIZE = "50m"  # Yambda size: '50m' or '500m'
NO_DRIFT_CHECK = False  # True skips the re-scored ndcg@10 vs record check

# fixed Kaggle paths
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"

import os
import time

# train.max_hours is measured from here (the session start), not from the script start
os.environ.setdefault("HOOKREC_SESSION_START", str(int(time.time())))
print("HOOKREC_SESSION_START =", os.environ["HOOKREC_SESSION_START"])

In [ ]:
# Clone the repo and install it. The image's torch is used (no torch install).
import importlib
import os
import shutil
import subprocess
import sys


def _stream(cmd):
    """Run cmd, streaming its output; raise on a non-zero exit (a `!cmd` magic would not)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{' '.join(cmd[:5])} ... exited {p.returncode}")


if os.path.exists(REPO):
    shutil.rmtree(REPO)
token = ""
try:  # optional, only for a private repo: Add-ons -> Secrets -> GITHUB_TOKEN (attached)
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
except Exception:
    token = ""
url = REPO_URL
if token and REPO_URL.startswith("https://"):
    url = "https://x-access-token:" + token + "@" + REPO_URL[len("https://"):]
r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", GIT_REF, url, REPO], capture_output=True, text=True
)
if r.returncode != 0:
    log = r.stdout + r.stderr
    if token:
        log = log.replace(token, "***")
    raise RuntimeError(
        f"git clone {REPO_URL} -b {GIT_REF} failed (exit {r.returncode}):\n{log}\n"
        "Check REPO_URL / GIT_REF and that Internet is ON; a private repo needs the "
        "GITHUB_TOKEN secret attached to this notebook."
    )
# the clone is saved with the output: never leave the token in .git/config
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
del token, url
_stream([sys.executable, "-m", "pip", "install", "-q", "-e", REPO])
sys.path.insert(0, os.path.join(REPO, "src"))
importlib.invalidate_caches()
SHA = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print(f"hookrec {REPO_URL} {GIT_REF} @ {SHA}")

In [ ]:
# Environment + input discovery (inputs are found by marker files; no slugs to edit).
import os
import sys
from pathlib import Path

from hookrec.utils.kaggle import (
    check_disk, env_report, find_inputs, find_one, find_processed, has_run, run_logged,
)

env_report(WORK)

PROCESSED = find_processed(SIZE, INPUT_ROOT)
RESULT_INPUTS = find_inputs("_HOOKREC_RESULTS_*", INPUT_ROOT)
CKPT_INPUTS = find_inputs("_HOOKREC_CKPT", INPUT_ROOT)
if not RESULT_INPUTS:
    find_one("_HOOKREC_RESULTS_*", INPUT_ROOT, hint="Attach the K3 and/or K4 outputs.")
print("processed:", PROCESSED)
print("results:", RESULT_INPUTS)
print("checkpoints:", CKPT_INPUTS)

In [ ]:
import json
import shutil
import sys

from hookrec.eval.report import write_readme_tables

RESULTS = Path(OUT, "results")
RESULTS.mkdir(parents=True, exist_ok=True)
for d in RESULT_INPUTS:
    for p in sorted([*d.glob("*.json"), *d.glob("*.npz")]):
        if not (RESULTS / p.name).exists():
            shutil.copyfile(p, RESULTS / p.name)

records = []
for p in sorted(RESULTS.glob("*.json")):
    try:
        records.append((p, json.loads(p.read_text())))
    except ValueError:
        print("skipping unreadable", p.name)
records.sort(key=lambda pr: pr[1].get("timestamp", ""))
tests, evals = {}, {}
for p, r in records:
    ex = r.get("extra") or {}
    if ex.get("split") != "test" or ex.get("processed_dir") != PROCESSED.name:
        continue
    if ex.get("kind") == "eval" or r["run_name"].endswith("-eval"):
        evals[ex.get("source_run")] = r["timestamp"]
    else:
        tests[r["run_name"]] = (p, r)  # timestamp order: the latest record wins


def ckpt_for(fname, model):
    # prefer the K4 output that wrote this record, then any output with a final model
    homes = [d.parent for d in RESULT_INPUTS if (d / fname).exists()]
    homes += [d.parent.parent for d in CKPT_INPUTS if d.name == model]
    for h in homes:
        c = h / "ckpt" / model / "test"
        if (c / "model_final.pt").exists():
            return c
    return None


FAILED = []
for name, (p, r) in sorted(tests.items()):
    if evals.get(name, "") >= r["timestamp"]:
        print(f"{name}: already evaluated, skipping")
        continue
    cmd = [
        sys.executable, f"{REPO}/scripts/run_eval.py", "--processed", PROCESSED, "--run", p,
        "--out", RESULTS,
    ]
    if "model" in r["config"]:
        ck = ckpt_for(p.name, name.removesuffix("-test"))
        if ck is None:
            FAILED.append(f"{name}: no ckpt/<name>/test/model_final.pt; attach the K4 output that trained it")
            print(FAILED[-1])
            continue
        cmd += ["--ckpt", ck]
    if NO_DRIFT_CHECK:
        cmd.append("--no-drift-check")
    try:
        run_logged(cmd, cwd=REPO)
    except RuntimeError as e:  # evaluate the rest, then fail the version below
        FAILED.append(f"{name}: {e}")

REPORT = write_readme_tables(RESULTS, Path(OUT, "reports", "results.md"))
print(REPORT.read_text())
if FAILED:
    raise RuntimeError(f"{len(FAILED)} evaluation(s) failed:\n" + "\n".join(FAILED))

In [ ]:
check_disk(WORK)
for p in sorted(Path(OUT).rglob("*")):
    if p.is_file() and "eval" in p.name:
        print(p.relative_to(OUT))